# 04 — CNN-only Model
Single-timestep CNN + Sensor MLP + Fusion + 3 risk heads (no LSTM yet). Validates the spatial branch of the architecture before adding temporal modeling.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from dataset import PatchDataset, load_processed_field, train_val_test_split
from models import CNNRiskModel, MultiTaskRiskLoss

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

PROCESSED_DIR = "../data/processed"
FIELD_ID = sorted(os.listdir(PROCESSED_DIR))[0]


## Build a single-timestep patch dataset (most recent date)

In [ ]:
import pandas as pd

dates, patches_by_date, sensors, coords = load_processed_field(FIELD_ID, PROCESSED_DIR)
patches = patches_by_date[-1]                                  # (N, C, H, W)
sensor_row = sensors[-1] if sensors is not None else np.zeros(8, dtype=np.float32)
sensor_feats = np.tile(sensor_row, (len(patches), 1)).astype(np.float32)

labels_df = pd.read_csv(f"../data/labels/{FIELD_ID}.csv", parse_dates=["timestamp"])
y_row = labels_df[["stress_risk", "water_risk", "pest_risk"]].iloc[-1].values.astype(np.float32)
labels = np.tile(y_row, (len(patches), 1))

train_idx, val_idx, test_idx = train_val_test_split(len(patches))

train_ds = PatchDataset(patches[train_idx], sensor_feats[train_idx], labels[train_idx])
val_ds = PatchDataset(patches[val_idx], sensor_feats[val_idx], labels[val_idx])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
print(f"train: {len(train_ds)}  val: {len(val_ds)}")


## Model, loss, optimizer

In [ ]:
in_channels = patches.shape[1]
sensor_features = sensor_feats.shape[1]

model = CNNRiskModel(in_channels=in_channels, sensor_features=sensor_features).to(device)
criterion = MultiTaskRiskLoss(weights=(1.0, 1.0, 1.0))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)


## Training loop

In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    with torch.set_grad_enabled(train):
        for batch in loader:
            img = batch["image"].to(device)
            sensor = batch["sensor"].to(device)
            label = batch["label"].to(device)

            preds = model(img, sensor)
            loss, _ = criterion(preds, label)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * img.size(0)
    return total_loss / len(loader.dataset)


N_EPOCHS = 20
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, N_EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"epoch {epoch:02d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")


## Loss curves

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("CNN-only model — training curve")
plt.show()


## Save checkpoint

In [ ]:
os.makedirs("../data/processed/checkpoints", exist_ok=True)
torch.save(
    {"model_state_dict": model.state_dict(), "in_channels": in_channels, "sensor_features": sensor_features},
    "../data/processed/checkpoints/cnn_baseline.pt",
)
print("saved checkpoint")
